In [10]:
from itertools import combinations
from conda_build import features
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore', category=UserWarning)


df=pd.read_csv("Database/Dataset_85/copy.csv")
features = ['sredni_czas','mediana_czas','max_czas','skladalnosc','obecnosc','n_600ns','n_1us','stddevi']
results=[]
df=df.dropna(subset=features).reset_index(drop=True)
X_all_scaled = MinMaxScaler().fit_transform(df[features])
scaled_dict = {f: X_all_scaled[:, i] for i, f in enumerate(features)}


for subset in combinations(features, 3):
    try:
        # 1. Extract and drop any rows with missing data for these 3 factors
        X_scaled = np.array([scaled_dict[f] for f in subset]).T

        # 3. Fit KMedoids
        model = KMeans(n_clusters=3, random_state=42, max_iter=2500)
        labels = model.fit_predict(X_scaled)

        #PCA - w grupach
        pca=PCA(n_components=3)
        pca.fit(X_scaled)
        pc_res=np.abs(pca.components_[0])

        # 4. Check if we have at least 2 clusters
        if len(np.unique(labels)) > 1:
            score = silhouette_score(X_scaled, labels)
            results.append({
            'subset': subset,
            'silhouette_score': score,
            'PC1': pc_res[0],
            'PC2': pc_res[1],
            'PC3': pc_res[2]
        })

    except Exception as e:
        print(f"Error with factors {subset}: {e}")
        continue

# PCA - Gobal
pca_g=PCA(n_components=3)
pca_g.fit(X_all_scaled)
pcg_res=np.abs(pca_g.components_)
wynik_pcg=pd.DataFrame({
    'factori': features,
    'PC1': pcg_res[0],
    'PC2': pcg_res[1],
    'PC3': pcg_res[2]

})
wynik=pd.DataFrame(results)
wynik=wynik.sort_values(by='silhouette_score', ascending=False).reset_index(drop=True)
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    print(wynik)
    print(wynik_pcg)

                                      subset  silhouette_score       PC1  \
0      (sredni_czas, mediana_czas, max_czas)          0.692256  0.484294   
1          (mediana_czas, max_czas, stddevi)          0.674662  0.608700   
2          (mediana_czas, max_czas, n_600ns)          0.673706  0.646329   
3            (mediana_czas, max_czas, n_1us)          0.664585  0.656900   
4       (sredni_czas, mediana_czas, stddevi)          0.634302  0.556580   
5                 (max_czas, n_600ns, n_1us)          0.631199  0.769872   
6           (mediana_czas, n_600ns, stddevi)          0.611925  0.796100   
7       (sredni_czas, mediana_czas, n_600ns)          0.608121  0.519375   
8             (mediana_czas, n_600ns, n_1us)          0.603158  0.752088   
9         (sredni_czas, mediana_czas, n_1us)          0.603151  0.529237   
10            (mediana_czas, n_1us, stddevi)          0.600701  0.816137   
11          (sredni_czas, max_czas, n_600ns)          0.600649  0.522668   
12          

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans
from sklearn_extra.cluster import KMedoids
from sklearn.metrics import silhouette_score

# importy danych - tu można zmieniać
df=pd.read_csv('Database/Dataset_85/copy.csv')
etykiety=['sredni_czas', 'mediana_czas', 'max_czas', 'stddevi']


df=df.dropna(subset=etykiety).reset_index(drop=True)
rzeczy=df[etykiety]
#normalizacja
norm=MinMaxScaler().fit_transform(rzeczy)

# tabela
tab=pd.DataFrame(norm, columns=etykiety)

def dobor_k (x):
    for k in range(2, 6):
        model_KMeans=KMeans(n_clusters=k, random_state=42).fit(x)
        score_KMeans=silhouette_score(x, model_KMeans.labels_)
        model_KMedo = KMedoids(n_clusters=k, random_state=42).fit(x)
        score_KMedo = silhouette_score(x, model_KMedo.labels_)
        print(f"KMedoids Algo: For k={k}, Silhouette Score: {score_KMedo:.3f}")
        print(f"KMean Algo: For k={k}, Silhouette Score: {score_KMeans:.3f}")

dobor_k(norm)

KMedoids Algo: For k=2, Silhouette Score: 0.634
KMean Algo: For k=2, Silhouette Score: 0.633
KMedoids Algo: For k=3, Silhouette Score: 0.660
KMean Algo: For k=3, Silhouette Score: 0.661
KMedoids Algo: For k=4, Silhouette Score: 0.575
KMean Algo: For k=4, Silhouette Score: 0.610
KMedoids Algo: For k=5, Silhouette Score: 0.551
KMean Algo: For k=5, Silhouette Score: 0.562


In [2]:
import torch
x=torch.randn(100,20)
print(x)

tensor([[-0.4160, -0.1374, -0.1707,  ..., -0.1081, -0.5759,  0.1614],
        [ 0.1780, -1.3889, -1.4519,  ..., -0.1312,  0.7646, -0.7989],
        [-0.3041,  0.8454, -1.5221,  ..., -1.5138, -0.8102,  2.6629],
        ...,
        [-0.8769,  0.2942, -0.0644,  ...,  0.7049,  0.2925,  0.6104],
        [-0.8078,  0.8431, -0.6972,  ..., -1.0290,  0.1760, -2.9550],
        [-0.8808, -0.4682,  1.1496,  ..., -0.8196, -0.6174, -1.1052]])
